<a href="https://colab.research.google.com/github/wiz124/chem169-git/blob/main/Li_Harry_RID_020_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
#Exercise 0
import numpy as np
import pandas as pd
import h5py
import re
import time
import warnings
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import AgglomerativeClustering, DBSCAN
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from scipy.sparse.csgraph import connected_components
from scipy.sparse import csr_matrix
import matplotlib.pyplot as plt

HDF5_PATH  = 'otnhfK06zM'
EXCEL_PATH = 'mtb_with_localization.xlsx'

with h5py.File(HDF5_PATH, 'r') as f:
    protein_ids = list(f.keys())
    embeddings_raw = np.array([f[pid][:] for pid in protein_ids], dtype=np.float32)

df = pd.read_excel(EXCEL_PATH)

def extract_location(text):
    """Pull the first location term from a raw UniProt subcellular-location string."""
    if pd.isna(text):
        return None
    m = re.search(r'SUBCELLULAR LOCATION:\s*([^{;.]+)', text)
    return m.group(1).strip() if m else None

df['loc'] = df['Subcellular location [CC]'].apply(extract_location)

# Keep only the 5 most common classes to avoid tiny-class issues
top_classes = df['loc'].value_counts().head(5).index.tolist()
df_labeled  = df[df['loc'].isin(top_classes)].copy()

id_to_idx = {pid: i for i, pid in enumerate(protein_ids)}
mask      = df_labeled['Entry'].isin(id_to_idx)
df_use    = df_labeled[mask].copy()

indices    = [id_to_idx[e] for e in df_use['Entry']]
embeddings = embeddings_raw[indices]

le     = LabelEncoder()
labels = le.fit_transform(df_use['loc'])
t0 = time.time()
similarity_matrix = cosine_similarity(embeddings)
similarity_matrix = np.clip(similarity_matrix, 0, 1)
print(f'done in {time.time()-t0:.2f}s')
print(f'Shape: {similarity_matrix.shape}')
print(f'Similarity range: [{similarity_matrix.min():.3f}, {similarity_matrix.max():.3f}]')

/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


done in 0.03s
Shape: (1025, 1025)
Similarity range: [0.019, 1.000]


In [15]:
#Exercise 1
def cluster_connected_components(sim_matrix, threshold=0.8):
    adjacency = (sim_matrix >= threshold).astype(int)
    np.fill_diagonal(adjacency, 0)
    n_comps, comp_labels = connected_components(csr_matrix(adjacency), directed=False)
    return comp_labels, n_comps

THRESHOLD = 0.8

t0 = time.time()
cc_labels, cc_n_clusters = cluster_connected_components(similarity_matrix, threshold=THRESHOLD)
cc_runtime = time.time() - t0

cc_sizes      = np.bincount(cc_labels)
cc_largest    = int(cc_sizes.max())
cc_singletons = int((cc_sizes == 1).sum())

print(f'  Number of clusters: {cc_n_clusters}')
print(f'  Largest cluster   : {cc_largest} proteins')
print(f'  Singleton clusters: {cc_singletons}')
print(f'  Runtime           : {cc_runtime:.4f} s')

  Number of clusters: 509
  Largest cluster   : 214 proteins
  Singleton clusters: 414
  Runtime           : 0.0228 s


In [16]:
#Exercise 2
distance_matrix = 1 - similarity_matrix
distance_matrix = np.clip(distance_matrix, 0, None)
np.fill_diagonal(distance_matrix, 0)

AGG_THRESHOLD = 0.2   # tune: try 0.1, 0.15, 0.2, 0.3

t0 = time.time()
agg = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=AGG_THRESHOLD,
    metric='precomputed',
    linkage='average'
)
agg_labels  = agg.fit_predict(distance_matrix)
agg_runtime = time.time() - t0

agg_n_clusters  = len(np.unique(agg_labels))
agg_sizes       = np.bincount(agg_labels)
agg_largest     = int(agg_sizes.max())
agg_singletons  = int((agg_sizes == 1).sum())

print('=== Agglomerative Clustering ===')
print(f'  Distance threshold : {AGG_THRESHOLD}')
print(f'  Number of clusters : {agg_n_clusters}')
print(f'  Largest cluster    : {agg_largest} proteins')
print(f'  Singleton clusters : {agg_singletons}')
print(f'  Runtime            : {agg_runtime:.4f} s')

thresholds  = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]
agg_results = []
for thr in thresholds:
    m = AgglomerativeClustering(n_clusters=None, distance_threshold=thr,
                                metric='precomputed', linkage='average')
    lbl = m.fit_predict(distance_matrix)
    agg_results.append({'threshold': thr, 'n_clusters': len(np.unique(lbl))})

pd.DataFrame(agg_results)

DBSCAN_EPS         = 0.2   # tune: try 0.1, 0.15, 0.2, 0.3
DBSCAN_MIN_SAMPLES = 2

t0 = time.time()
db            = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMPLES, metric='precomputed')
db_labels_raw = db.fit_predict(distance_matrix)
db_runtime    = time.time() - t0

n_noise        = int((db_labels_raw == -1).sum())
db_n_named     = len(np.unique(db_labels_raw[db_labels_raw != -1]))

# Assign each noise point its own singleton cluster
db_labels = db_labels_raw.copy()
next_id   = db_labels.max() + 1
for i, lbl in enumerate(db_labels):
    if lbl == -1:
        db_labels[i] = next_id
        next_id += 1

db_sizes      = np.bincount(db_labels)
db_largest    = int(db_sizes.max())
db_singletons = int((db_sizes == 1).sum())

print('=== DBSCAN ===')
print(f'  eps               : {DBSCAN_EPS}')
print(f'  min_samples       : {DBSCAN_MIN_SAMPLES}')
print(f'  Named clusters    : {db_n_named}  (excl. noise)')
print(f'  Noise points      : {n_noise}  (→ singletons)')
print(f'  Total clusters    : {len(np.unique(db_labels))}')
print(f'  Largest cluster   : {db_largest} proteins')
print(f'  Runtime           : {db_runtime:.4f} s')
eps_values     = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]
dbscan_results = []
for eps in eps_values:
    lbl = DBSCAN(eps=eps, min_samples=2, metric='precomputed').fit_predict(distance_matrix)
    dbscan_results.append({'eps': eps,
                            'n_clusters': len(np.unique(lbl[lbl != -1])),
                            'n_noise'   : int((lbl == -1).sum())})

pd.DataFrame(dbscan_results)

=== Agglomerative Clustering ===
  Distance threshold : 0.2
  Number of clusters : 639
  Largest cluster    : 18 proteins
  Singleton clusters : 465
  Runtime            : 0.0391 s
=== DBSCAN ===
  eps               : 0.2
  min_samples       : 2
  Named clusters    : 95  (excl. noise)
  Noise points      : 414  (→ singletons)
  Total clusters    : 509
  Largest cluster   : 214 proteins
  Runtime           : 0.0555 s


,eps,n_clusters,n_noise
0,0.05,48,901
1,0.10,91,752
2,0.15,120,594
3,0.20,95,414
4,0.25,51,278
5,0.30,27,144
6,0.40,2,30
7,0.50,1,2


In [18]:
#Exercise 3
def cluster_aware_split(cluster_labels, test_frac=0.2, random_state=42):

    rng = np.random.default_rng(random_state)
    unique_clusters = np.unique(cluster_labels)
    rng.shuffle(unique_clusters)

    n_total   = len(cluster_labels)
    test_idx  = []
    train_idx = []

    for c in unique_clusters:
        members = np.where(cluster_labels == c)[0].tolist()
        if len(test_idx) / max(n_total, 1) < test_frac:
            test_idx.extend(members)
        else:
            train_idx.extend(members)

    return np.array(train_idx), np.array(test_idx)

def evaluate_clustering(name, cluster_labels, embeddings, labels,
                         n_clusters, largest_cluster, runtime):
    train_idx, test_idx = cluster_aware_split(cluster_labels)

    if len(test_idx) == 0 or len(np.unique(labels[test_idx])) < 2:
        print(f'  {name}: bad split – no usable test set')
        return {'Method': name, '# Clusters': n_clusters,
                'Largest Cluster': largest_cluster,
                'Runtime (s)': round(runtime, 4), 'Test Accuracy': 'N/A'}

    X_train, y_train = embeddings[train_idx], labels[train_idx]
    X_test,  y_test  = embeddings[test_idx],  labels[test_idx]

    clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    clf.fit(X_train, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test))

    print(f'  {name}: train={len(train_idx)}, test={len(test_idx)}, accuracy={acc:.4f}')
    return {'Method': name, '# Clusters': n_clusters,
            'Largest Cluster': largest_cluster,
            'Runtime (s)': round(runtime, 4),
            'Test Accuracy': round(acc, 4)}

results = [
    evaluate_clustering('Connected Components', cc_labels,  embeddings, labels,
                         cc_n_clusters, cc_largest, cc_runtime),
    evaluate_clustering('Agglomerative',        agg_labels, embeddings, labels,
                         agg_n_clusters, agg_largest, agg_runtime),
    evaluate_clustering('DBSCAN',               db_labels,  embeddings, labels,
                         len(np.unique(db_labels)), db_largest, db_runtime),
]
results_df = pd.DataFrame(results,
    columns=['Method', '# Clusters', 'Largest Cluster', 'Runtime (s)', 'Test Accuracy'])
print(results_df.to_string(index=False))
results_df

  Connected Components: train=643, test=382, accuracy=0.7618
  Agglomerative: train=820, test=205, accuracy=0.7122
  DBSCAN: train=820, test=205, accuracy=0.7220
              Method  # Clusters  Largest Cluster  Runtime (s)  Test Accuracy
Connected Components         509              214       0.0228         0.7618
       Agglomerative         639               18       0.0391         0.7122
              DBSCAN         509              214       0.0555         0.7220


,Method,# Clusters,Largest Cluster,Runtime (s),Test Accuracy
0,Connected Components,509,214,0.0228,0.7618
1,Agglomerative,639,18,0.0391,0.7122
2,DBSCAN,509,214,0.0555,0.7220


Exercise 3

Q1: The clustering method does not significantly affect accuracy. The accuracy between all three are within 4%

Q2: connected components is the fastest

Q3: Agglomerative

#Exercise 4

Q1: Agglomerative gave the most clusters. This is because each data point starts as its own cluster and is then further refined by merging points based on certain metrics. This results in lower cluster sizes and thus more clusters.

Q2: Algorithm choice didn't affect accuracy that much. This might be because of the low training set size or threshold. This tells me that quality and quantity of the training set matters more than the algorithm.

Q3: I would do connected components because the algorithm works similarity to comparing sequence similarity and grouping them together. This makes the algorithm fast and easy to understand.